# 第3回　ばらつきを測る／正規分布という「仮定」

統計学Ⅰ（B）　／　北星学園大学

今日も**▶を上から押すだけ**。注目するのは ――

> 平均が同じでも、**ばらつき**が違えば、まったく別のデータだ。

In [ ]:
# 準備：ライブラリと北辰大学データ。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("準備OK")

---
## フック：平均が同じ2つのクラス

2つのクラスがテストを受けた。**どちらも平均58点**。あなたは「同じようなクラスだ」と思うだろうか？

- クラスA：全員がだいたい58点
- クラスB：半分が20点、半分が96点

平均だけ見ると区別できない。グラフにしてみよう。

In [ ]:
クラスA = np.full(30, 58.0)                 # 全員ほぼ平均点
クラスB = np.array([20]*15 + [96]*15, float)  # 二極化

print(f"クラスA  平均 {クラスA.mean():.0f}  標準偏差 {クラスA.std(ddof=1):.1f}")
print(f"クラスB  平均 {クラスB.mean():.0f}  標準偏差 {クラスB.std(ddof=1):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
ax[0].hist(クラスA, bins=range(0,101,5), color="#80cbc4", edgecolor="white"); ax[0].set_title("クラスA（平均58）")
ax[1].hist(クラスB, bins=range(0,101,5), color="#e8503a", edgecolor="white"); ax[1].set_title("クラスB（平均58）")
for a in ax: a.set_xlabel("点"); a.axvline(58, color="gray", ls="--")
plt.show()

**平均は同じ58点。でも中身は正反対。** 平均だけでは、この違いは絶対に見えない。

足りないのは「**どれだけ散らばっているか**」＝ばらつきの情報だ。

---
## ばらつきを測る：分散と標準偏差

ばらつきは「各データが平均からどれだけ離れているか」で測る。

1. 各データの **平均からの差**（偏差）を出す
2. そのままだとプラスとマイナスで打ち消し合う → **二乗**してから平均する ＝ **分散**
3. 二乗したままだと単位が点²で直感的でない → **ルートを取って元の単位に戻す** ＝ **標準偏差(SD)**

北辰大の `テスト点` で計算してみよう。

In [ ]:
t = df["テスト点"]
print(f"平均　　　 {t.mean():.1f} 点")
print(f"分散　　　 {t.var():.1f} （点の二乗・直感的でない）")
print(f"標準偏差SD {t.std():.1f} 点（元の単位に戻った・これを使う）")

plt.figure(figsize=(8,4))
plt.hist(t, bins=25, color="#80cbc4", edgecolor="white")
m, s = t.mean(), t.std()
plt.axvline(m, color="#1565c0", lw=2, label=f"平均 {m:.0f}")
plt.axvspan(m-s, m+s, color="#e8503a", alpha=0.15, label=f"平均±1SD（{m-s:.0f}〜{m+s:.0f}）")
plt.xlabel("テスト点"); plt.ylabel("人数"); plt.legend(); plt.title("標準偏差＝平均からの『標準的な散らばり幅』")
plt.show()

---
## 偏差値の正体

「偏差値」は、点数を **平均50・標準偏差10** のものさしに置き直したもの。

$$ 偏差値 = 50 + 10 \times \frac{あなたの点 - 平均}{標準偏差} $$

つまり「平均からSD何個分ずれているか」を 50 中心に表しただけ。テストが難しくて平均が低くても、自分の**相対位置**が分かる。

In [ ]:
def 偏差値(点, データ):
    return 50 + 10 * (点 - データ.mean()) / データ.std()

for 点 in [80, 57, 35]:
    print(f"テスト点 {点:3d} → 偏差値 {偏差値(点, t):.1f}")
print("\n平均(57点)ちょうどなら偏差値はほぼ50。+1SD(約69点)で偏差値60。")

---
## 正規分布と 68-95-99.7 則

左右対称で釣鐘型の分布を **正規分布** という。多くの自然なばらつきが近い形になる。正規分布なら、ばらつきの目安が決まっている：

- 平均 ±1SD に約 **68%**
- 平均 ±2SD に約 **95%**
- 平均 ±3SD に約 **99.7%**

テスト点が本当にこの通りか、実データで数えてみよう。

In [ ]:
m, s = t.mean(), t.std()
for k, 理論 in [(1,68),(2,95),(3,99.7)]:
    実際 = ((t>=m-k*s)&(t<=m+k*s)).mean()*100
    print(f"平均±{k}SD に 実際 {実際:.1f}%  （理論 {理論}%）")

x = np.linspace(t.min(), t.max(), 200)
plt.figure(figsize=(8,4))
plt.hist(t, bins=25, density=True, color="#80cbc4", edgecolor="white", label="実データ")
plt.plot(x, stats.norm.pdf(x, m, s), color="#e8503a", lw=2, label="正規分布（平均とSDから）")
plt.xlabel("テスト点"); plt.ylabel("割合"); plt.legend(); plt.title(f"テスト点はほぼ正規分布（歪度 {t.skew():.2f}）")
plt.show()

ほぼ理論通り（67% / 95% / 99.8%）。テスト点は正規分布で**よく近似できる**。

だが――**世の中のすべてが正規分布ではない**。

In [ ]:
inc = df["世帯年収万円"]
x = np.linspace(inc.min(), 3000, 300)
plt.figure(figsize=(8,4))
plt.hist(inc[inc<3000], bins=40, density=True, color="#80cbc4", edgecolor="white", label="実データ（年収）")
plt.plot(x, stats.norm.pdf(x, inc.mean(), inc.std()), color="#e8503a", lw=2, label="正規分布を当てはめると…")
plt.xlabel("世帯年収（万円）"); plt.ylabel("割合"); plt.legend()
plt.title(f"年収は正規分布で表せない（歪度 {inc.skew():.1f}・右に裾）")
plt.show()
print(f"テスト点の歪度 {t.skew():.2f}（≒対称）  vs  年収の歪度 {inc.skew():.1f}（強く右に歪む）")

赤い正規分布の曲線は、年収のデータにまるで合っていない。**「正規分布だと仮定する」のは判断であって、自然法則ではない。** 歪んだデータに正規分布の道具（68-95-99.7則など）を当てると、間違った結論になる。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 分散 | 平均からの差を二乗して平均（単位は元の二乗） |
| 標準偏差(SD) | 分散のルート。**ばらつきの標準的な幅**（元の単位） |
| 偏差値 | 平均50・SD10 に置き直した相対位置 |
| 68-95-99.7則 | 正規分布なら±1/2/3SDに68/95/99.7%（**正規のときだけ**） |

> **平均は分布の『位置』、SDは『幅』。両方見て初めてデータが分かる。**
> そして正規分布は便利な仮定だが、当てはまるかは**自分で確かめる**もの。

**課題（Moodle）**：平均が同じでSDが違う2データの解釈／偏差値の読み方。